In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===============================
# datasets
# ===============================
FILES = {
    "AWR_steady": "AWR_steady.csv",
    "AWR_unsteady": "AWR_unsteady.csv",
    "IWR_steady": "IWR_steady.csv",
    "IWR_unsteady": "IWR_unsteady.csv",
}

# ===============================
# methods
# ===============================
METHODS = [
    "mmw_HR_dtw",
    "mmw_HR_drscc",
    "mmw_HR_xcorr",
    "mmw_HR_gccphat",
    "mmw_HR_peaks_drift",
]

# ===============================
# metric function
# ===============================
def compute_metrics(gt, pred):

    err = pred - gt
    err = err[~np.isnan(err)]

    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    bias = np.mean(err)
    std = np.std(err, ddof=1)

    n = len(err)
    ci95 = 1.96 * std / np.sqrt(n)

    return mae, rmse, bias, std, ci95


# ===============================
# function to compute table
# ===============================
def compute_table(df):

    gt = df["ECG_HR"].values
    rows = []

    for method in METHODS:

        pred = df[method].values

        mae, rmse, bias, std, ci = compute_metrics(gt, pred)

        rows.append({
            "Method": method.replace("mmW_HR_", ""),
            "MAE": round(mae, 3),
            "RMSE": round(rmse, 3),
            "Bias": round(bias, 3),
            "Std": round(std, 3),
            "95% CI": round(ci, 3)
        })

    return pd.DataFrame(rows)


# ===============================
# run analysis
# ===============================
all_tables = {}
all_dfs = []

for dataset, file in FILES.items():

    df = pd.read_csv(file)

    table = compute_table(df)

    all_tables[dataset] = table
    all_dfs.append(df)


# ===============================
# combined dataset
# ===============================
df_all = pd.concat(all_dfs, ignore_index=True)
all_tables["ALL_DATA"] = compute_table(df_all)


# ===============================
# print results
# ===============================
for name, table in all_tables.items():

    print("\n============================")
    print(name)
    print("============================")
    print(table.to_string(index=False))


AWR_steady
            Method   MAE  RMSE  Bias   Std  95% CI
        mmw_HR_dtw 5.020 7.401 0.309 7.469   2.070
      mmw_HR_drscc 4.959 7.144 1.499 7.055   1.956
      mmw_HR_xcorr 4.959 7.144 1.499 7.055   1.956
    mmw_HR_gccphat 4.484 5.714 0.145 5.771   1.600
mmw_HR_peaks_drift 4.865 6.954 1.212 6.917   1.917

AWR_unsteady
            Method    MAE   RMSE    Bias    Std  95% CI
        mmw_HR_dtw 22.646 26.801 -17.335 20.570   4.508
      mmw_HR_drscc 24.759 28.835 -18.842 21.966   4.813
      mmw_HR_xcorr 24.759 28.835 -18.842 21.966   4.813
    mmw_HR_gccphat 27.333 30.958 -24.930 18.471   4.048
mmw_HR_peaks_drift 24.687 28.940 -21.812 19.140   4.194

IWR_steady
            Method    MAE   RMSE   Bias    Std  95% CI
        mmw_HR_dtw 45.724 50.662 43.443 26.174   4.683
      mmw_HR_drscc 43.257 48.925 37.889 31.083   5.561
      mmw_HR_xcorr 43.257 48.925 37.889 31.083   5.561
    mmw_HR_gccphat 45.781 51.476 42.748 28.798   5.153
mmw_HR_peaks_drift 44.057 49.628 39.811 29.75